# Python Analytics
Анализ данных: pandas, requests, matplotlib

In [ ]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import csv
import time

print('pandas:', pd.__version__)
print('requests:', requests.__version__)

## 1. DataFrame — основы

In [ ]:
# Создаём DataFrame из словаря
data = {
    'имя': ['Никита', 'Анна', 'Иван', 'Мария'],
    'город': ['Пермь', 'Москва', 'Пермь', 'Екатеринбург'],
    'возраст': [19, 25, 31, 22],
    'зарплата': [35000, 90000, 60000, 75000]
}

df = pd.DataFrame(data)
df

In [ ]:
# Осмотр таблицы
print('Размер:', df.shape)
df.info()

In [ ]:
# Статистика по числовым столбцам
df.describe()

In [ ]:
# Выбор столбцов
print(df['имя'])          # один столбец -> Series
print('---')
df[['имя', 'зарплата']]  # несколько -> DataFrame

In [ ]:
# Выбор строк
print(df.iloc[0])   # по порядковому номеру
print('---')
print(df.loc[1])    # по индексу

In [ ]:
# Сохранить и загрузить CSV
df.to_csv('сотрудники.csv', index=False, encoding='utf-8')
df2 = pd.read_csv('сотрудники.csv')
df2

## 2. Фильтрация, сортировка, группировка

In [ ]:
# Одно условие
df[df['возраст'] > 20]

In [ ]:
# И (&) — оба условия
df[(df['возраст'] > 20) & (df['зарплата'] > 60000)]

In [ ]:
# ИЛИ (|) — хотя бы одно
df[(df['город'] == 'Пермь') | (df['зарплата'] > 80000)]

In [ ]:
# Сортировка по убыванию зарплаты
df.sort_values('зарплата', ascending=False)

In [ ]:
# Средняя зарплата по городам
print(df.groupby('город')['зарплата'].mean())
print('---')
# Количество сотрудников в каждом городе
print(df['город'].value_counts())

## 3. Очистка данных

In [ ]:
# DataFrame с пропусками
data_dirty = {
    'имя': ['Никита', 'Анна', 'Иван', 'Мария', 'Олег'],
    'город': ['Пермь', 'Москва', None, 'Екатеринбург', 'Пермь'],
    'возраст': [19, 25, 31, None, 22],
    'зарплата': [35000, 90000, 60000, 75000, None]
}

df_dirty = pd.DataFrame(data_dirty)
df_dirty

In [ ]:
# Сколько пропусков
print(df_dirty.isnull().sum())

In [ ]:
# Заполнить пропуски
df_clean = df_dirty.fillna({'город': 'Неизвестно', 'возраст': 0, 'зарплата': 0})
df_clean.to_csv('сотрудники_clean.csv', index=False, encoding='utf-8')
df_clean

In [ ]:
# apply — применить функцию к столбцу
print(df_dirty['зарплата'].apply(lambda x: x * 1.1))  # +10%
print('---')
print(df_dirty['имя'].apply(lambda x: x.upper()))     # верхний регистр

## 4. Requests — работа с API

In [ ]:
# Текущая погода в Перми через wttr.in API
r = requests.get('https://wttr.in/Perm?format=j1')
data = r.json()

current = data['current_condition'][0]
print('Температура:', current['temp_C'], '°C')
print('Ощущается как:', current['FeelsLikeC'], '°C')
print('Влажность:', current['humidity'], '%')
print('Описание:', current['weatherDesc'][0]['value'])

In [ ]:
# Прогноз на 3 дня -> DataFrame
rows = []
for day in data['weather']:
    rows.append({
        'дата': day['date'],
        'макс_темп': int(day['maxtempC']),
        'мин_темп': int(day['mintempC']),
    })

df_weather = pd.DataFrame(rows)
df_weather

In [ ]:
# Пагинация — сбор данных с нескольких страниц
url = 'https://jsonplaceholder.typicode.com/posts'
all_posts = []

for page in range(1, 4):
    r = requests.get(url, params={'_page': page, '_limit': 10})
    posts = r.json()
    all_posts.extend(posts)
    print(f'Страница {page}: получено {len(posts)} постов')
    time.sleep(0.5)

df_posts = pd.DataFrame(all_posts)
df_posts.to_csv('posts.csv', index=False, encoding='utf-8')
print(f'Сохранено: {len(df_posts)} строк')
df_posts.head()

## 5. Визуализация

In [ ]:
# График прогноза температуры в Перми
df_weather.plot(
    x='дата',
    y=['макс_темп', 'мин_темп'],
    kind='line',
    marker='o',
    title='Прогноз температуры в Перми',
    ylabel='Температура °C',
    figsize=(8, 4)
)

plt.tight_layout()
plt.savefig('weather_chart.png')
plt.show()